# SDFT — Self-Distillation Fine-Tuning: Qwen2.5-0.5B-Instruct (TRL backend)

In [ ]:
from aligntune.core.backend_factory import create_distill_trainer

# privileged_context_column="input": CuratorKIT's canonical DataSample schema
# (id/instruction/input/output/chosen/rejected/...) has no "context"/"hint"/
# "feedback"/etc field for _apply_privileged_context's alias scan to match, so
# without this override the dataset ends up with zero rows containing
# "privileged_context" and training silently completes at global_step=0 (no
# error). Alpaca's "input" field (extra context alongside the instruction) is
# exactly what SDFT/SDPO mean by privileged context, so point it there
# explicitly.
trainer = create_distill_trainer(
    student_model="Qwen/Qwen2.5-0.5B-Instruct",
    dataset_name="tatsu-lab/alpaca",
    split="train",
    backend="unsloth",
    output_dir="./out_sdft",
    batch_size=1,
    num_epochs=1,
    max_steps=10,
    learning_rate=5e-5,
    teacher_model_kind="base",
    num_generations=1,
    max_completion_length=64,
    max_seq_length=128,
    max_samples=256,
    privileged_context_column="input",
    use_peft=True,
    lora_r=8,
    loggers=["none"],
    seed=42,
    eval_strategy="no",
)

results = trainer.train()
print("SDFT (self-distillation fine-tuning) training completed.")
print(results)